In [23]:
%pip install mariadb

Note: you may need to restart the kernel to use updated packages.


In [24]:
import mariadb
# Execption used to stop the Notebook cell execution politely
class StopExecution(Exception):
    def _render_traceback_(self):
        pass
# Connect to the server and return a Connection object for the db_name database in our case it's mydb.

def connectToDB(db_name):
    try:
        return mariadb.connect(
            user="root",
            password="root",
            host="localhost", 
# Use of localhost because 1/ we don't haveacces to the docker local network 2/ we have made a port redirection from themariadb server to the host
            port=3306,
            database=db_name
        )
    except mariadb.Error as e:
        print(f"Error connecting to MariaDB Platform: {e}")
        raise StopExecution


conn = connectToDB("flight_reservation")


In [25]:
# A bit of dark magic? Just two intrincated query, one to get the TABLES, one
#for each TABLE to get the COLUMNS.
# And a clever use of format to have aligned columns.
def showTablesAndColumns(conn):
    cur = conn.cursor()
    # Query the tables
    cur.execute("SHOW TABLES;")
    # For each table
    for table in cur.fetchall():
        print("**",table[0],"**")

        # Defining the output format {:16} indicate at least 16 characters,# add spaces if necessary

        outputFormat="{:16}\t{:10}\t{}\t{}\t{}\t{}"
        #Printing the headers
        print(outputFormat.format("Field","Type","Null","Key","Default","Extra"))
        # Query the columns
        cur.execute('SHOW COLUMNS FROM {};'.format(table[0]))
        # For each column
        for attribute in cur.fetchall():
        # Print the attribute (*attribute give the content of the tuple to the format function)
            print(outputFormat.format(*attribute))

showTablesAndColumns(conn)


** Booking **
Field           	Type      	Null	Key	Default	Extra
id_booking      	int(11)   	NO	PRI	None	auto_increment
client_name     	varchar(50)	NO		None	
quantity        	int(11)   	NO		None	
id_flight       	varchar(20)	NO	MUL	None	
** Flight **
Field           	Type      	Null	Key	Default	Extra
id_flight       	varchar(20)	NO	PRI	None	
maximum_capacity	int(11)   	NO		None	
current_capacity	int(11)   	NO		None	


In [ ]:
def addFlight(id_flight, maximum_capacity):
    try:
        conn = connectToDB("flight_reservation")
        cur = conn.cursor()

        # insertion sécurisée
        cur.execute(
            "INSERT INTO Flight (id_flight, maximum_capacity, current_capacity) VALUES (?, ?, ?)",
            (id_flight, maximum_capacity, 0)
        )

        conn.commit()  # IMPORTANT pour sauvegarder
        print("Flight ajouté avec succès")

        cur.close()
        conn.close()

    except mariadb.Error as e:
        print(f"Erreur : {e}")

#addFlight("FL2811", 10)

Erreur : Duplicate entry 'FL2811' for key 'PRIMARY'


In [27]:
import time
import random

def addBooking(id_flight, client_name, quantity):
    try:
        conn = connectToDB("flight_reservation")
        cur = conn.cursor()

        #line added for 6.4
        cur.execute("SET TRANSACTION ISOLATION LEVEL SERIALIZABLE")

        # 1. Vérifier si le vol existe
        cur.execute("SELECT COUNT(*) FROM Flight WHERE id_flight=?;", (id_flight,))
        if cur.fetchone()[0] == 0:
            conn.close()
            return (False, "Invalid")

        # 2. Vérifier capacité disponible
        cur.execute("""
            SELECT (current_capacity + ?) <= maximum_capacity 
            FROM Flight WHERE id_flight=?;
        """, (quantity, id_flight))

        if not cur.fetchone()[0]:
            conn.close()
            return (False, "Full")

        # 3. Simulation paiement
        time.sleep(random.randint(3,5))

        # 4. Ajouter réservation
        cur.execute("""
            INSERT INTO Booking (client_name, quantity, id_flight)
            VALUES (?, ?, ?);
        """, (client_name, quantity, id_flight))

        # 5. Mettre à jour capacité
        cur.execute("""
            UPDATE Flight 
            SET current_capacity = current_capacity + ?
            WHERE id_flight=?;
        """, (quantity, id_flight))

        conn.commit()
        conn.close()

        return (False, "Reserved")

    except Exception as e:
        print("Error:", e)
        if conn:
            conn.rollback()
            conn.close()
        return (True, "Error")

In [28]:
def cleanDB(id_flight):
    conn = connectToDB("flight_reservation")
    cur = conn.cursor()
    cur.execute("DELETE FROM Booking WHERE id_flight=?;",(id_flight,))
    cur.execute("UPDATE Flight SET current_capacity=0 WHERE id_flight=?;",(id_flight,))
    conn.commit() # Without commit the modification are not applied to the database


In [29]:
def processBooking(id_flight, client_name, quantity):
    print("Processing Flight",id_flight,"for",client_name,":", quantity,"seats.")
    retry,status = addBooking(id_flight,client_name,quantity)
    print(client_name,":",status, "Should retry:",retry)
    while retry:
        print("Retry Flight", id_flight, "for", client_name, ":", quantity, "seats.")
        retry, status = addBooking(id_flight, client_name, quantity)
        print(client_name, ":", status)

In [30]:
import threading
import random
import time

In [31]:
# Test parameters
id_flight="LH6795"
base_name="Client "
# Clean the db for the test
cleanDB(id_flight)
# Creating several Thread representing several client that try to booksimultaneously a random number of seat
jobs=[]
for i in range(5): 
    jobs.append(threading.Thread(target=processBooking,args=(id_flight,base_name+str(i), random.randint(1,4))))

# Start each job, giving 3s of delay between each
for job in jobs:
    job.start()
    time.sleep(3)
# Wait for all the thread to finish
for job in jobs:
    job.join()
print("Finished")

OperationalError: Lock wait timeout exceeded; try restarting transaction